# 14. Paired contrasts, uncertainty, and decision thresholds

![Paired inference](../images/14_paired_inference.svg)

**Learning goals:** form independent participant-level contrasts, compute Student $t$ and bootstrap intervals, respect nested trials with a hierarchical bootstrap, estimate effect resolution and noncentral-$t$ power, test superiority and equivalence, and apply Holm correction.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

SEED = 14
rng = np.random.default_rng(SEED)
np.set_printoptions(precision=3, suppress=True)
print(f"NumPy {np.__version__}; seed={SEED}")

## 1. Collapse paired trials to independent participant contrasts

Participant $i$ has baseline and intervention measurements. The independent analysis and sampling unit is the participant-level difference $D_i=\bar Y_{i1}-\bar Y_{i0}$, not each trial row. Positive participant-level contrasts in this simulation mean improvement. Shared participant and trial noise make pairing useful because those components cancel in the difference. If treatment were randomized, the experimental unit would instead be the unit receiving an independent assignment.

In [ ]:
n_participants, n_trials = 24, 10
participant_effect = rng.normal(0, 0.50, size=(n_participants, 1))
shared_trial_noise = rng.normal(0, 0.25, size=(n_participants, n_trials))
baseline = participant_effect + shared_trial_noise + rng.normal(0, 0.12, (n_participants, n_trials))
true_improvement = 0.18
intervention = (participant_effect + shared_trial_noise + true_improvement
                + rng.normal(0, 0.12, (n_participants, n_trials)))
differences = intervention.mean(axis=1) - baseline.mean(axis=1)
assert differences.shape == (n_participants,)
print(f"raw rows={2*n_participants*n_trials}; independent paired analysis units={n_participants}")
print(f"mean participant contrast={differences.mean():.3f}")

## 2. Student $t$ interval

The standard error is $s_D/\sqrt n$, where `ddof=1` uses the sample-variance denominator $n-1$. If participant differences are iid normal, the resulting statistic has an exact $t_{n-1}$ distribution. With independent but nonnormal differences, the interval is an asymptotic approximation whose quality can suffer under small samples, strong skew, or outliers. In repeated sampling under the assumptions, a 95% procedure covers the fixed mean difference in about 95% of studies; it is not a 95% posterior probability for this one interval.

In [ ]:
def t_interval(values, confidence=0.95):
    values = np.asarray(values, dtype=float)
    n = len(values)
    mean = values.mean()
    se = values.std(ddof=1) / np.sqrt(n)
    critical = stats.t.ppf((1 + confidence) / 2, df=n - 1)
    return mean, se, (mean - critical * se, mean + critical * se)

mean_d, se_d, interval_t = t_interval(differences)
reference = stats.ttest_rel(intervention.mean(axis=1), baseline.mean(axis=1))
skewness = stats.skew(differences, bias=False)
q1, q3 = np.quantile(differences, [.25, .75])
iqr = q3 - q1
outlier_count = int(np.sum((differences < q1 - 1.5 * iqr) | (differences > q3 + 1.5 * iqr)))
assert np.isclose(reference.statistic, mean_d / se_d)
assert np.isfinite(skewness) and 0 <= outlier_count <= n_participants
print(f"mean={mean_d:.3f}, SE={se_d:.3f}, 95% t interval={interval_t}")
print(f"difference skewness={skewness:.3f}; Tukey outliers={outlier_count}")

## 3. Percentile and hierarchical bootstraps

The participant bootstrap resamples the already-computed $D_i$ values. Comparing its percentile interval with the $t$ interval is a diagnostic, not proof that either procedure has perfect coverage. The hierarchical bootstrap first resamples participants and then resamples paired trial indices within each sampled participant. Paired trial indices are drawn together so shared within-trial structure is preserved. Vectorizing the simple bootstrap is fast; the nested version uses loops for clarity and remains small.

In [ ]:
def participant_bootstrap(values, B, rng):
    idx = rng.integers(0, len(values), size=(B, len(values)))
    return values[idx].mean(axis=1)

def hierarchical_bootstrap(base, inter, B, rng):
    P, T = base.shape
    estimates = np.empty(B)
    for b in range(B):
        sampled_people = rng.integers(0, P, size=P)
        person_differences = np.empty(P)
        for j, pid in enumerate(sampled_people):
            trial_idx = rng.integers(0, T, size=T)
            person_differences[j] = (inter[pid, trial_idx] - base[pid, trial_idx]).mean()
        estimates[b] = person_differences.mean()
    return estimates

boot_rng = np.random.default_rng(SEED + 1)
simple_boot = participant_bootstrap(differences, B=5000, rng=boot_rng)
hier_boot = hierarchical_bootstrap(baseline, intervention, B=1500, rng=boot_rng)
simple_ci = np.quantile(simple_boot, [.025, .975])
hier_ci = np.quantile(hier_boot, [.025, .975])
assert simple_ci[0] < simple_ci[1] and hier_ci[0] < hier_ci[1]
endpoint_gap = np.max(np.abs(simple_ci - np.asarray(interval_t)))
print("participant bootstrap 95% interval:", simple_ci.round(3))
print("hierarchical bootstrap 95% interval:", hier_ci.round(3))
print(f"largest participant-bootstrap versus t endpoint gap={endpoint_gap:.3f}")

## 4. Effect resolution and prospective power

The confidence-interval half-width $h=t^*s_D/\sqrt n$ is a direct resolution measure. Under an iid normal-difference model, a prospective paired $t$ statistic follows a noncentral $t$ distribution with standardized effect $\delta=\mu_D/\sigma_D$ and noncentrality $\lambda=\delta\sqrt n$. Power is probability beyond the two critical tails. For nonnormal differences, this is a model-based approximation.

In [ ]:
def paired_t_power(n, standardized_effect, alpha=0.05):
    df = n - 1
    critical = stats.t.ppf(1 - alpha / 2, df)
    ncp = standardized_effect * np.sqrt(n)
    return stats.nct.cdf(-critical, df, ncp) + stats.nct.sf(critical, df, ncp)

critical_95 = stats.t.ppf(.975, n_participants - 1)
half_width = critical_95 * se_d
assumed_effect = 0.5
power = paired_t_power(n_participants, assumed_effect)
assert 0 <= power <= 1
print(f"95% half-width={half_width:.3f}")
print(f"power for standardized effect {assumed_effect:.2f}: {power:.3f}")

## 5. Superiority, equivalence, and Holm correction

Positive superiority requires a one-sided lower confidence bound above zero. Equivalence with margin $\Delta$ uses two one-sided tests, or TOST: the two-sided 90% interval must fit entirely inside $(-\Delta,\Delta)$ at level 0.05. Failure to find superiority does not prove equivalence. For several predeclared hypotheses, Holm compares ordered p-values sequentially and controls family-wise error.

In [ ]:
alpha = 0.05
one_sided_lower = mean_d - stats.t.ppf(1 - alpha, n_participants - 1) * se_d
superior = one_sided_lower > 0
equivalence_margin = 0.25
_, _, interval_90 = t_interval(differences, confidence=0.90)
equivalent = interval_90[0] > -equivalence_margin and interval_90[1] < equivalence_margin

def holm_adjust(p_values):
    p = np.asarray(p_values, dtype=float)
    order = np.argsort(p)
    scaled = (len(p) - np.arange(len(p))) * p[order]
    adjusted_sorted = np.minimum(1.0, np.maximum.accumulate(scaled))
    adjusted = np.empty_like(p)
    adjusted[order] = adjusted_sorted
    return adjusted

raw_p = np.array([0.004, 0.018, 0.041, 0.30])
adjusted_p = holm_adjust(raw_p)
assert np.all((0 <= adjusted_p) & (adjusted_p <= 1))
print(f"superior={superior}; 90% interval={np.round(interval_90, 3)}; equivalent={equivalent}")
print("Holm adjusted p-values:", adjusted_p.round(3))

fig, ax = plt.subplots(figsize=(7, 3))
ax.hist(simple_boot, bins=35, color="#69a7d0", edgecolor="white")
ax.axvline(0, color="#b74d4d", label="no effect")
ax.axvline(mean_d, color="#173f5f", label="observed mean")
ax.set(xlabel="mean paired contrast", ylabel="bootstrap count", title="Participant bootstrap distribution")
ax.legend()
plt.tight_layout()
plt.show()

## Exercises and takeaways

1. Increase participant count while holding the observed standard deviation fixed. How does half-width change?
2. Set the equivalence margin to 0.10. Why can superiority and equivalence conclusions differ?
3. Independently resample baseline and intervention trial indices in the hierarchical bootstrap. Which covariance is destroyed?
4. Verify Holm adjusted p-values by sorting and applying cumulative maxima.

**Brief answers:** half-width decreases approximately as $1/\sqrt n$. Superiority asks whether the effect is positive, while equivalence asks whether its magnitude is practically small. Independent trial resampling destroys within-trial covariance. Holm's cumulative maximum ensures adjusted p-values cannot decrease as raw ordered p-values increase.

**Takeaway:** valid paired inference begins with independent unit-level differences. Interval type, bootstrap hierarchy, decision margin, and multiplicity family must be chosen for the scientific claim.

## Continue learning

[Previous notebook: 13](13_context_interventions.ipynb) | [Lecture](../lectures/14_paired_inference.md) | [Curriculum](../README.md) | [Next notebook: 15](15_exposure_and_replication.ipynb)